# 02 — ML Modeling & LLM Explainability

**Objectives**
1. Compare 4 binary classifiers (Dropout=1 vs Graduate=0) via stratified 5-fold cross-validation.
2. Tune the top 2 candidates with Optuna.
3. Select the final model and evaluate it **once** on the held-out test set.
4. SHAP analysis — global importance + individual explanations.
5. *(Phase 3, later section)* LLM risk explanations for currently-enrolled students.

See `docs/ADR_001_binary_classification.md` for the binary-classification rationale and
`docs/DATA_INSIGHTS.md` for the EDA-derived feature signal map and the two dropout profiles.

In [ ]:
import sys
sys.stdout.reconfigure(encoding="utf-8")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src.data.loader import DataLoader
from src.data.preprocessor import Preprocessor
from src.models.classifiers import (
    LogisticRegressionModel,
    RandomForestModel,
    GradientBoostingModel,
    HistGradientBoostingModel,
)
from src.models.trainer import ModelTrainer
from src.models.evaluator import ModelEvaluator

PROJECT_ROOT = Path.cwd().parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
ARTIFACTS = PROJECT_ROOT / "artifacts"

RANDOM_STATE = 42
print("Project root:", PROJECT_ROOT)
print("Processed data:", DATA_PROCESSED)
print("Artifacts:", ARTIFACTS)

## 1. Load Data

In [ ]:
loader = DataLoader(DATA_PROCESSED)
X_train, X_test, y_train, y_test = loader.load_train_test()
enrolled_demo = loader.load_enrolled_demo()

# Reset positional index so every downstream array (y_proba, SHAP values)
# aligns by position with these DataFrames/Series.
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print("X_train:", X_train.shape, "| X_test:", X_test.shape, "| enrolled_demo:", enrolled_demo.shape)
print("\nClass balance (train, normalized):")
display(y_train.value_counts(normalize=True).rename({0: "Graduate (0)", 1: "Dropout (1)"}))
print("Class balance (test, normalized):")
display(y_test.value_counts(normalize=True).rename({0: "Graduate (0)", 1: "Dropout (1)"}))

## 2. Step A — Baseline Comparison (4 Models)

All four models address the ~61/39 class imbalance uniformly through the
`AbstractBaseModel` Template Method: Logistic Regression and Random Forest use
`class_weight='balanced'`, while Gradient Boosting and Hist Gradient Boosting
receive a balanced `sample_weight` array at fit time. `ModelTrainer` calls
`model.fit(...)` identically for all four — each CV fold fits its own cloned
preprocessor to prevent leakage.

In [ ]:
preprocessor = Preprocessor().build_transformer()
trainer = ModelTrainer(preprocessor, n_splits=5, random_state=RANDOM_STATE)

models = [
    LogisticRegressionModel(),
    RandomForestModel(),
    GradientBoostingModel(),
    HistGradientBoostingModel(),
]

step_a_results = trainer.compare_models(models, X_train, y_train)
display(step_a_results.round(4))

## 3. Step B — Tuning Top 2 Candidates

From Step A, two models are excluded from tuning:
- **Random Forest** — notably lower ROC-AUC (calibration gap) despite competitive F1.
- **Logistic Regression** — limited tuning headroom (a single hyperparameter, `C`).

**Gradient Boosting** and **Hist Gradient Boosting** are statistically tied on F1 and
both have strong ROC-AUC, so both are tuned with Optuna (30 trials, optimizing CV mean F1).

In [ ]:
tuning_candidates = [GradientBoostingModel(), HistGradientBoostingModel()]

step_b_tuned = {}        # model_name -> full tuned CV metrics dict
step_b_best_params = {}  # model_name -> best_params dict

for model in tuning_candidates:
    print(f"Tuning {model.name} (30 trials)...")
    tuned = trainer.tune_with_optuna(model, X_train, y_train, n_trials=30, metric="f1")
    best_params = tuned["best_params"]
    step_b_best_params[model.name] = best_params
    print(f"  -> best_params: {best_params}")
    step_b_tuned[model.name] = trainer.cross_validate(
        model, X_train, y_train, params=best_params
    )

# Before/after table: Step A default vs Step B tuned
default_lookup = step_a_results.set_index("model_name")
rows = []
for name, tuned_cv in step_b_tuned.items():
    rows.append({
        "model_name": name,
        "default_f1_mean": default_lookup.loc[name, "f1_mean"],
        "tuned_f1_mean": tuned_cv["f1_mean"],
        "f1_delta": tuned_cv["f1_mean"] - default_lookup.loc[name, "f1_mean"],
        "default_roc_auc_mean": default_lookup.loc[name, "roc_auc_mean"],
        "tuned_roc_auc_mean": tuned_cv["roc_auc_mean"],
        "roc_auc_delta": tuned_cv["roc_auc_mean"] - default_lookup.loc[name, "roc_auc_mean"],
    })
before_after = pd.DataFrame(rows)
display(before_after.round(4))

## 4. Final Model Selection

**Tiebreak rule:** select the model with the higher tuned mean F1. If the F1
difference between the two tuned candidates is below 0.002 (within noise), break
the tie using the higher tuned mean ROC-AUC.

In [ ]:
gbm_name = "Gradient Boosting"
hist_name = "Hist Gradient Boosting"

gbm_f1 = step_b_tuned[gbm_name]["f1_mean"]
hist_f1 = step_b_tuned[hist_name]["f1_mean"]

if abs(hist_f1 - gbm_f1) < 0.002:
    # Tie on F1 -> break by ROC-AUC
    winner_name = max(step_b_tuned, key=lambda n: step_b_tuned[n]["roc_auc_mean"])
    justification = (
        f"F1 tie (|{hist_f1:.4f} - {gbm_f1:.4f}| < 0.002) -> tiebreak on ROC-AUC; "
        f"{winner_name} wins with roc_auc_mean={step_b_tuned[winner_name]['roc_auc_mean']:.4f}"
    )
else:
    winner_name = max(step_b_tuned, key=lambda n: step_b_tuned[n]["f1_mean"])
    justification = (
        f"{winner_name} has the higher tuned f1_mean "
        f"({step_b_tuned[winner_name]['f1_mean']:.4f})"
    )

model_registry = {gbm_name: GradientBoostingModel, hist_name: HistGradientBoostingModel}
final_model = model_registry[winner_name]()
final_params = step_b_best_params[winner_name]

print("Final model:", winner_name)
print("Best params:", final_params)
print("Justification:", justification)

## 5. Step C — Final Fit & One-Time Test Evaluation

The winner is refit on the **full** training set, then evaluated on the held-out
test set. This is the **first and only** time `X_test`/`y_test` are used in the
entire project — `evaluate_on_test` is the single method permitted to touch them.

In [ ]:
evaluator = ModelEvaluator(ARTIFACTS)
fitted_pipeline = evaluator.fit_final_model(
    final_model, final_params, preprocessor, X_train, y_train
)
test_results = evaluator.evaluate_on_test(fitted_pipeline, X_test, y_test)

print(f"Test F1        : {test_results['f1']:.4f}")
print(f"Test ROC-AUC   : {test_results['roc_auc']:.4f}")
print(f"Test Precision : {test_results['precision']:.4f}")
print(f"Test Recall    : {test_results['recall']:.4f}")

print("\nClassification report (test set):")
display(pd.DataFrame(test_results["classification_report"]).T.round(4))

In [ ]:
cm_path = evaluator.plot_confusion_matrix(test_results["confusion_matrix"], "confusion_matrix.png")
y_proba = fitted_pipeline.predict_proba(X_test)[:, 1]
roc_path = evaluator.plot_roc_curve(y_test, y_proba, "roc_curve.png")

display(Image(filename=str(cm_path)))
display(Image(filename=str(roc_path)))

In [ ]:
model_path = evaluator.save_model(fitted_pipeline, "best_model.pkl")
size_kb = model_path.stat().st_size / 1024
print(f"Saved fitted pipeline -> {model_path} ({size_kb:.1f} KB)")

## 6. Metrics Consolidation (Offline vs Online)

A single CSV records the full evaluation story, tagged by `evaluation_type`:
- **`offline_cv_default`** — Step A, all 4 models at default hyperparameters (cross-validated).
- **`offline_cv_tuned`** — Step B, the 2 tuned candidates (cross-validated).
- **`online_test`** — Step C, the winning model on the held-out test set.

This offline (cross-validation) vs online (held-out) distinction satisfies the
project's dual-evaluation requirement in one artifact.

In [ ]:
rows = []

# Step A — offline CV, default params, all 4 models
for _, r in step_a_results.iterrows():
    rows.append({
        "model_name": r["model_name"],
        "evaluation_type": "offline_cv_default",
        "f1": r["f1_mean"],
        "roc_auc": r["roc_auc_mean"],
        "precision": r["precision_mean"],
        "recall": r["recall_mean"],
    })

# Step B — offline CV, tuned params, top 2
for name, cv in step_b_tuned.items():
    rows.append({
        "model_name": name,
        "evaluation_type": "offline_cv_tuned",
        "f1": cv["f1_mean"],
        "roc_auc": cv["roc_auc_mean"],
        "precision": cv["precision_mean"],
        "recall": cv["recall_mean"],
    })

# Step C — online held-out test, winner only
rows.append({
    "model_name": final_model.name,
    "evaluation_type": "online_test",
    "f1": test_results["f1"],
    "roc_auc": test_results["roc_auc"],
    "precision": test_results["precision"],
    "recall": test_results["recall"],
})

metrics_path = evaluator.save_metrics_csv(rows, "metrics_comparison.csv")
print("Saved metrics ->", metrics_path)
display(pd.DataFrame(rows).round(4))

## 7. Step D — SHAP Analysis

SHAP explains the final model on the post-preprocessing feature space.
- **Background:** a 150-row sample of `X_train` (`random_state=42`).
- **Explain:** the full `X_test` (726 rows).

`compute_shap_values` prints shape diagnostics. For this tree-based model the
explainer typically returns the 3D form `(samples, features, classes)`, so the
`ndim == 3` -> class-1 (Dropout) slice branch is expected to fire — unlike the
LogisticRegression-based unit tests, which returned the 2D form.

In [ ]:
X_background = X_train.sample(150, random_state=RANDOM_STATE)
shap_result = evaluator.compute_shap_values(fitted_pipeline, X_background, X_test)

summary_path = evaluator.shap_summary_plot(shap_result, "shap_summary.png")
display(Image(filename=str(summary_path)))

## 8. Individual Explanations — Dry Run for Phase 3

Validates that `explain_instance()` output is sensible before building
`StudentRiskExplainer`. Per `DATA_INSIGHTS.md`, we expect features like
`approval_rate_sem2`, `Curricular units 2nd sem (approved)`,
`Tuition fees up to date`, and `Debtor` to surface among the top contributors —
and ideally to see both academic (Profile A) and financial (Profile B) signals
across different students.

In [ ]:
# y_proba (from cell 13) is positionally aligned with X_test / shap_result.
mask_dropout = (y_test.values == 1)
mask_graduate = (y_test.values == 0)

idx_high_risk = int(np.where(mask_dropout)[0][np.argmax(y_proba[mask_dropout])])
idx_low_risk = int(np.where(mask_graduate)[0][np.argmin(y_proba[mask_graduate])])
idx_borderline = int(np.argmin(np.abs(y_proba - 0.5)))

cases = {
    "HIGH RISK (true Dropout, highest p)": idx_high_risk,
    "LOW RISK (true Graduate, lowest p)": idx_low_risk,
    "BORDERLINE (p closest to 0.5)": idx_borderline,
}

for label, row_idx in cases.items():
    exp = evaluator.explain_instance(shap_result, row_idx, fitted_pipeline, X_test, top_k=5)
    print("=" * 70)
    print(f"{label}  [X_test position {row_idx}]")
    print(f"  Predicted: {exp['prediction']}  |  P(Dropout) = {exp['probability_dropout']:.3f}")
    print(f"  True label: {'Dropout' if y_test.iloc[row_idx] == 1 else 'Graduate'}")
    print("  Top 5 features:")
    for f in exp["top_features"]:
        print(f"    {f['direction']:<16} {f['feature']:<45} "
              f"shap={f['shap_value']:+.3f}  value={f['feature_value']:.3f}")
print("=" * 70)

## 9. Summary & Phase 3 Preview

**Final model:** _(Hist Gradient Boosting, per the selection above)_ — refit on the
full training set with the Optuna-tuned hyperparameters.

**Held-out test performance:** see Section 5 (F1, ROC-AUC, precision, recall) and the
confusion-matrix / ROC plots.

**SHAP findings:** see Sections 7–8. Confirm that the strongest contributors match the
`DATA_INSIGHTS.md` signal map (academic engagement via `approval_rate_sem2` /
2nd-semester approved units, plus financial signals like `Tuition fees up to date` and
`Debtor`). The presence of financial features among top contributors for at least some
high-risk students indicates the model captures the non-academic dropout profile
(Profile B), not academics alone.

**Artifacts written:** `best_model.pkl`, `metrics_comparison.csv`, `confusion_matrix.png`,
`roc_curve.png`, `shap_summary.png`.

**Next (Phase 3):** apply this same `fitted_pipeline` + SHAP machinery to
`enrolled_demo.parquet` (the 794 currently-enrolled students with no ground truth) through
the `StudentRiskExplainer`, producing counselor-facing natural-language risk reports.